In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

# ---------------------------------------------------------
# CELL 1: Environment & LLM Setup
# ---------------------------------------------------------
# Load secrets from the .env file
load_dotenv()

# Strict check to ensure pipeline fails immediately if auth is missing
if os.environ.get("GOOGLE_API_KEY"):
    print("API Key is set. Ready to connect.")
else:
    raise ValueError("GOOGLE_API_KEY is not set. Check your .env file.")

# Initialize the base Gemini model. 
# We use temperature=0 to stop the agent from hallucinating tool names.
llm_gemini = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

API Key is set. Ready to connect.


In [9]:
from langchain.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun, ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import ArxivAPIWrapper, WikipediaAPIWrapper

# ---------------------------------------------------------
# CELL 2: Defining the Tools (The Agent's API Hooks)
# ---------------------------------------------------------
# The @tool decorator tells LangChain to convert this standard Python
# function into an AI-readable JSON schema.
# CRITICAL: The docstrings ("""...""") are the system prompts for the tools.

@tool
def search_duckduckgo(query: str) -> str:
    """Searches the latest news on DuckDuckGo for the given query and returns the results."""
    duck_search = DuckDuckGoSearchRun()
    return duck_search.invoke(query)

@tool
def arxiv_tool(query: str) -> str:
    """Queries the arXiv database for academic and scientific research papers."""
    arxiv_query = ArxivQueryRun(api_wrapper=ArxivAPIWrapper())
    return arxiv_query.invoke(query)

@tool
def wiki_tool(query: str):
    """Searches Wikipedia for general information on a given topic."""
    wiki_query = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
    return wiki_query.invoke(query)

@tool
def personal_info(name: str):
    """Use this tool to get personal information about Alice, Bob, or Charlie."""
    info = {
        "Alice": "Alice is a software engineer with 5 years of experience in AI.",
        "Bob": "Bob is a data scientist who loves working with large datasets.",
        "Charlie": "Charlie is a product manager with a background in tech startups."
    }
    return info.get(name, "No information available for this person.")

# ---------------------------------------------------------
# BINDING: Upgrading the LLM
# ---------------------------------------------------------
tools = [search_duckduckgo, arxiv_tool, wiki_tool, personal_info]

# This creates a NEW version of the LLM that knows these tools exist.
llm_with_tools = llm_gemini.bind_tools(tools)

In [ ]:
from typing import TypedDict, List
from langchain_core.messages import AnyMessage, SystemMessage

# ---------------------------------------------------------
# CELL 3: The Graph State & The LLM Node
# ---------------------------------------------------------

# 1. THE STATE: This is our agent's memory bank. 
# Instead of storing just a "name" or "text", it stores a growing list 
# of conversation messages (Human, AI, and Tool messages).
class graph_schema(TypedDict):
    messages: List[AnyMessage]

# 2. THE BRAIN: This function acts as the core reasoning engine.
def llm_node(state: graph_schema) -> graph_schema:
    # READ: Grab the current conversation history from the state
    messages = state['messages']

    # We create a system message to give the agent its instructions
    system_prompt = SystemMessage(
        content="You are a helpful assistant that can use tools to answer questions. Always use tools when you need external information."
    )

    # PROCESS: We pass the system prompt AND the entire message history to our tool-enabled LLM.
    # Because we use `llm_with_tools` here, the AI might output a standard text message, 
    # OR it might output a special "Tool Call" request.
    ai_response = llm_with_tools.invoke([system_prompt] + messages)

    # UPDATE: Append the AI's latest response to our list of messages
    state['messages'] = messages + [ai_response]

    # RETURN: Pass the updated state to the next step in the graph
    return state

In [4]:
from langchain_core.messages import ToolMessage

# ---------------------------------------------------------
# CELL 4: The Tool Execution Node
# ---------------------------------------------------------
def tool_node(state: graph_schema) -> graph_schema:
    # 1. READ: Get the latest message from the history
    messages = state['messages']
    last_message = messages[-1]

    # 2. PREPARE: Create a dictionary map of our available tools 
    # Example: {"search_duckduckgo": <function search_duckduckgo>}
    tools_by_name = {tool.name: tool for tool in tools}
    tool_results = []

    # 3. EXECUTE: Loop through every tool the LLM requested in its last message
    for tool_call in last_message.tool_calls:
        # Find the correct python function using the requested name
        requested_tool = tools_by_name[tool_call["name"]]
        
        # Invoke the python function with the arguments the LLM provided
        observation = requested_tool.invoke(tool_call["args"])

        # 4. PACKAGE: Wrap the raw string output in a special ToolMessage object.
        # The tool_call_id is critical; it proves to the LLM which question this answer belongs to.
        tool_results.append(
            ToolMessage(content=str(observation), tool_call_id=tool_call["id"])
        )

    # 5. UPDATE: Append the newly generated ToolMessages to the state
    state['messages'] = messages + tool_results

    # 6. RETURN: Pass the updated state to the next node
    return state

In [5]:
# ---------------------------------------------------------
# CELL 5: The Conditional Router
# ---------------------------------------------------------
# This function does not modify the state. It only reads it 
# and returns a string to tell LangGraph where to go next.

def if_tool_call(state: graph_schema) -> str:
    # Read the very last message in the list
    last_message = state['messages'][-1]

    # Check if the LLM populated the 'tool_calls' attribute
    if last_message.tool_calls:
        return "tool_node"
    else:
        return "end"

In [6]:
from langgraph.graph import StateGraph, START, END

# ---------------------------------------------------------
# CELL 6: Building the Graph Structure
# ---------------------------------------------------------

# 1. Initialize the graph with our specific dictionary structure
graph = StateGraph(graph_schema)

# 2. Register our two Python functions as nodes
graph.add_node("llm_node", llm_node)
graph.add_node("tool_node", tool_node)

# 3. Define the flow (Edges)
# Rule A: Always start by passing the user's input to the LLM
graph.add_edge(START, "llm_node")

# Rule B: After the LLM runs, use our routing function to decide the next step
graph.add_conditional_edges(
    "llm_node", 
    if_tool_call,
    {"tool_node": "tool_node", "end": END}
)

# Rule C: If the tool node runs, ALWAYS send the data back to the LLM
graph.add_edge("tool_node", "llm_node")

# 4. Compile the graph
react_graph = graph.compile()

In [7]:
from langchain_core.messages import HumanMessage

# ---------------------------------------------------------
# CELL 7: Execution
# ---------------------------------------------------------

# Create the initial state payload
initial_payload = {"messages": [HumanMessage(content="Who is Alice and what is the latest news on AI?")]}

# Trigger the graph and capture the final state dictionary
final_state = react_graph.invoke(initial_payload)

# Print just the text content of the very last message in the list
print("\n--- FINAL AI RESPONSE ---")
print(final_state['messages'][-1].content)

Impersonate 'chrome_114' does not exist, using 'random'



--- FINAL AI RESPONSE ---
[{'type': 'text', 'text': "Alice is a software engineer with 5 years of experience in AI.\n\nRegarding the latest news on AI, current trends indicate a focus on AI Overview visibility for marketers, advancements in artificial intelligence, and the impact of AI on search engine optimization. There's also news and reviews on AI software, hardware, and research, including discussions on AI documentaries.", 'extras': {'signature': 'CrMBAb4+9vtk2jQs4c9xOTbe5pfTAgGPsSSBs2X+Wr5EECYJSzGP9LnzIKPv3xwtDd0qkeFcq/wzmfuibgkHT9qpfJvlDWSnHsBvcGWim/steGloRlyA6Etfuu8T0ZzuvoVVBQBoUX7wBYQbSlNE/Qr0dAHhfFIyWGLvnrt/IwPiuUkWzJ9EwmVJ0PRDjui617raAHl5+XszbioEVWCLe1r409tRxWNH2JTXGIM8evxNcCmHu8k='}}]
